# Surface quality — what the completion looks like

Point clouds turned into shaded meshes, plus the one diagnostic a mesh cannot
show. Numbers come first: the strongest result in this project, point density, is
invisible in a rendered surface.

Needs a GPU and the fold's checkpoints. Metrics live in `MSN_eval_metrics.ipynb`.

**Two hard rules.** These meshes are for looking at, never for computing a metric:
reconstruction moves the original points by a median 5.3 mm, the same order as
the model's own error. And comparing two models means identical reconstruction
parameters and camera — both are locked constants in `mesh_viz`, because every
knob changes how smooth a surface looks and tuning per figure would let a
parameter impersonate a model improvement.

Counter-intuitive but measured: more smoothing makes models differ *more*. At low
smoothing the reconstruction's own texture drowns the signal.

## 1 · Configuration

The four cells of the 2x2 at **fold 0** — because it is fold 0, not because it
looks best. All four validate the same twenty skulls.

In [ ]:
import json
import os
import sys

REPO = os.path.abspath(".")
while REPO != os.path.dirname(REPO) and not os.path.isdir(os.path.join(REPO, "src", "models")):
    REPO = os.path.dirname(REPO)
assert os.path.isdir(os.path.join(REPO, "src", "models")), f"repo root not found from {os.getcwd()}"
for sub in ("src/models", "src/eval", "src/data"):
    sys.path.insert(0, os.path.join(REPO, sub))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import paths

FOLD = 0
CONFIGS = ["cd_only", "lr_fix_only", "rep_w05", "cd_rep05_full"]
MODELS = [(f"{c}_f{FOLD}", f"msn_skullfix/{c}_f{FOLD}") for c in CONFIGS]

MESH_MODEL = f"cd_rep05_full_f{FOLD}"   # the configuration this project adopts
CAMERA = "defect"                       # faces the hole. "default" matches the older
                                        # figures in reports/ but is nearly orthogonal
                                        # to the defect direction, so a defective and a
                                        # complete skull render almost identically
SKULL = None        # None picks the skull closest to this fold's median, see section 3
N_STATS = 8         # skulls behind section 6's archived numbers; rendering uses one
DEVICE = "/GPU:0"   # "/CPU:0" if another kernel is holding the GPU

CACHE = os.path.join(REPO, paths.DATA_CACHE)
RAW_ROOT = os.path.join(REPO, paths.RAW_ROOT)
print("REPO =", REPO)

## 2 · Load and run inference

One model instance, weights swapped in a loop, and `model.predict` rather than
`model(x)`. Both avoid GPU leaks paid for once already.

In [ ]:
import importlib

import numpy as np
import tensorflow as tf
for g in tf.config.experimental.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)

import msn_skullfix as msn
import mesh_viz as mv
import report as rp
# Edited often, and without this you silently get the cached copy -- old figures,
# old numbers. Only these two: msn_skullfix defines Keras layers and hot-reloading
# it makes already-built models fail against the new classes.
importlib.reload(mv)
importlib.reload(rp)

data = np.load(CACHE)
ids, inputs, gt, scale_mm = data["ids"], data["inputs"], data["gt"], data["scale_mm"]

# One model is shared below, so all four have to be the same topology. Loading a
# checkpoint from another topology succeeds and draws the wrong picture: changing
# where the decoder's keys come from changes no weight's shape, so load_weights
# cannot catch it.
runs = {name: rp.Run(REPO, rel) for name, rel in MODELS}
archs = {r.arch_key for r in runs.values()}
assert len(archs) == 1, f"MODELS spans {sorted(r.arch_label for r in runs.values())}"
CFG = rp.arch_config(msn, archs.pop())
print("architecture:", next(iter(runs.values())).arch_label)

val_ids = runs[MODELS[0][0]].meta["val_ids"]
assert all(r.meta["val_ids"] == val_ids for r in runs.values()), "folds disagree on the split"

# Predict all twenty validation skulls, not just the ones the archive needs. It costs
# about a second each and it decouples two things that used to be tangled: which skull
# gets rendered, and which skulls the archived numbers cover. Setting SKULL by hand
# used to fail with a KeyError whenever it fell outside the statistics cohort.
val_pos = [i for i, s in enumerate(ids) if s in set(val_ids)]

# Cohort: the first N_STATS validation skulls IN `ids` ORDER. That is the cohort the
# eleven archived rows of surface_quality.csv were computed on, and fold 0 holds the
# same twenty skulls as the old single split, so the new rows stay comparable with
# them. Warning: roughness.csv, normal_quality.csv and p2s.csv take the first eight in
# `val_ids` order instead, which overlaps this set in exactly one skull. Never place a
# number from those beside a number from this table.
sel = val_pos[:N_STATS]
print(f"statistics cohort ({len(sel)}): {', '.join(ids[i] for i in sel)}")

preds = {}
x = [inputs[val_pos]]
if CFG.use_text:
    x.append(np.tile(np.load(os.path.join(REPO, paths.BERT_CACHE))[None], (len(val_pos), 1)))
with tf.device(DEVICE):
    model = msn.build_model(CFG)
    for name, _ in MODELS:
        model.load_weights(runs[name].weights)
        out = model.predict(x, batch_size=1, verbose=0)
        preds[name] = {i: out[j] for j, i in enumerate(val_pos)}
        print(f"  {name}: inference done")

## 3 · Numbers first

A picture is chosen and framed; these are not. The table is the frozen evaluation
from `eval_all_runs.csv`, not recomputed here, so it is the same number the thesis
quotes.

**Rendering only the best skull overstates the model**, so `SKULL = None` picks
the one nearest this fold's median.

In [ ]:
import pandas as pd

frozen = pd.read_csv(os.path.join(REPO, "experiments_log", "eval_all_runs.csv"),
                     dtype={"id": str})
frozen = frozen[(frozen["defect_def"] == "implant")
                & (frozen["run"].isin(runs))]
cols = ["defect_cov_mm", "CD_t_mm", "HD95_mm", "clump_%", "spacing_CV", "defect_n_pred"]
table = frozen.groupby("run", sort=False)[cols].mean().loc[list(runs)]
print(f"fold {FOLD}, mean over its {frozen['id'].nunique()} validation skulls\n")
print(table.round(3).to_string())
print("\nGround truth density for reference: clump_% = 0.0, spacing_CV = 0.145.")
print("Warning: repulsion is the whole point of the last row and it is invisible in")
print("every mesh below. That is what section 5 is for.")

per_skull = (frozen[frozen["run"] == MESH_MODEL]
             .set_index("id")["defect_cov_mm"].sort_values())
if SKULL is None:
    SKULL = str(per_skull.sub(per_skull.median()).abs().idxmin())
assert SKULL in val_ids, f"skull {SKULL} is not in fold {FOLD}'s validation set"
k_show = int(np.where(ids == SKULL)[0][0])

order = list(per_skull.index)
print(f"\n{MESH_MODEL} per skull, best to worst (mm):")
print("  best   " + "  ".join(f"{s}:{per_skull[s]:.2f}" for s in order[:3]))
print("  median " + "  ".join(f"{s}:{per_skull[s]:.2f}" for s in order[len(order) // 2 - 1:
                                                                     len(order) // 2 + 1]))
print("  worst  " + "  ".join(f"{s}:{per_skull[s]:.2f}" for s in order[-3:]))
print(f"\nrendering skull {SKULL} ({per_skull[SKULL]:.2f} mm, "
      f"rank {order.index(SKULL) + 1}/{len(order)})")

## 4 · Meshes

Ground truth against `MESH_MODEL`, two panels. This figure answers one question —
*is the completion a plausible skull with the hole filled* — and nothing else; the
four configurations differ in density, which no mesh can show.

In [ ]:
s_show = float(scale_mm[k_show])
items = [(mv.pc_to_mesh(gt[k_show], s_show), "ground truth"),
         (mv.pc_to_mesh(preds[MESH_MODEL][k_show], s_show), MESH_MODEL)]

# Makes the toolbar's camera icon export at 3x, so any angle you drag to can be
# saved as a PNG without touching code. Warning: that angle is not recorded
# anywhere -- for a reproducible one see 4.2.
SAVE_CFG = {"toImageButtonOptions": {"format": "png", "scale": 3,
                                     "filename": f"skull_{SKULL}_{MESH_MODEL}"}}

mv.fig_meshes(items, f"skull {SKULL} — surface comparison (reconstruction locked)",
              camera=CAMERA).show(config=SAVE_CFG)

### 4.1 · Four panels: raw volume, input, completion, raw volume

The only figure with the cost of the representation and the contribution of the
model in one frame. Outer panels are 0.475 mm voxels, inner two are ~4 mm points.

Warning: the outer panels are visibly crisper and **that gap is the
representation, not the model's quality**. 1 vs 2 is the cost of going to points,
2 vs 3 is the model, 3 vs 4 is the total. Reading 4 against 3 as *the model is
poor* is the one wrong way to look at it.

Warning: do not save this cell's output — four inline panels are 15-40 MB.

In [ ]:
import mesh_preview as mp
importlib.reload(mp)

LIGHT = dict(res=96)   # interactive only; the print version passes nothing and gets RECON
TRUTH_STEP = 4         # 1.90 mm for interaction; 2 (0.95 mm) for print


def four_panel(light=True, truth_step=None, camera=None):
    """Defective volume, input cloud, completion, complete volume."""
    kw = LIGHT if light else {}
    m_def, m_comp = mp.truth_meshes(SKULL, RAW_ROOT,
                                    truth_step or (TRUTH_STEP if light else 2))
    items = [(m_def, "truth: defective (nrrd, 0.475 mm)"),
             (mv.pc_to_mesh(inputs[k_show], s_show, **kw), "input (defective, 4096 pts)"),
             (mv.pc_to_mesh(preds[MESH_MODEL][k_show], s_show, **kw),
              f"completed — {MESH_MODEL}"),
             (m_comp, "truth: complete (nrrd, 0.475 mm)")]
    return mv.fig_meshes(items,
                         f"skull {SKULL} — outer panels are raw volumes; "
                         f"the crispness gap is the REPRESENTATION, not the model",
                         height=560, camera=camera or CAMERA)


four_panel().show(config=SAVE_CFG)

### 4.2 · Choosing an angle, and keeping it

Drag any figure and use the toolbar camera icon for a one-off; drag the widget
below and print x/y/z to get an angle you can add to `mesh_viz.CAMERAS` and reuse.

Warning: reading the angle back could not be verified from this side. If the print
still shows the starting value, use the toolbar icon.

In [ ]:
import plotly.graph_objects as go

_fig = mv.fig_meshes([(mv.pc_to_mesh(preds[MESH_MODEL][k_show], s_show, res=96),
                       f"{MESH_MODEL} — drag me")],
                     f"skull {SKULL} — pick an angle", height=520, camera=CAMERA)

# FigureWidget needs `anywidget` from plotly 6 on, and requirements-msn.txt pins
# ipywidgets but not that. Without it the figure is still draggable, but the
# camera cannot be read back into Python, so the next cell has nothing to print.
# This is the only cell in the notebook that needs it -- everything after 4.2,
# including the archive in section 6, runs either way.
try:
    picker = go.FigureWidget(_fig)
    display(picker)
except ImportError:
    picker = None
    print("anywidget is not installed, so the camera cannot be read back.\n"
          "Showing a static figure instead: drag it to look around, then set\n"
          "CAMERA to a key of mesh_viz.CAMERAS, or add an entry there by hand.\n"
          "`pip install anywidget` restores the read-back in the next cell.")
    _fig.show()

In [ ]:
if picker is None:
    print("skipped: no anywidget, so there is no camera to read back.\n"
          f"CAMERAS currently holds: {', '.join(mv.CAMERAS)}")
else:
    eye = picker.layout.scene.camera.eye
    print(f'    "my_view": dict(x={eye.x:.3f}, y={eye.y:.3f}, z={eye.z:.3f}),')
    print("\nAdd that line to CAMERAS in src/eval/mesh_viz.py, then set CAMERA = \"my_view\".")
    print(f"Warning: if this still reads {tuple(mv.CAMERAS[CAMERA].values())}, the drag did "
          f"not come back -- use the toolbar camera icon.")

### 4.3 · Print version

The locked `RECON`, `truth_step=2`, straight to PNG. Needs the headless Chrome from
section 5 of `setup_env.sh`, which lives on `/root` and is wiped by a redeploy.

In [ ]:
OUT_DIR = os.path.join(REPO, "reports", "preview")      # gitignored
os.makedirs(OUT_DIR, exist_ok=True)


def render_png(fig, name, height=700):
    p = os.path.join(OUT_DIR, name)
    fig.write_image(p, width=max(1200, 600 * len(fig.data)), height=height, scale=2)
    print(f"-> {os.path.relpath(p, REPO)}   {os.path.getsize(p) / 1e6:.2f} MB")
    return p


render_png(four_panel(light=False), f"skull_{SKULL}_{MESH_MODEL}_4panel.png")

## 5 · The diagnostic a mesh cannot give you

Nearest-neighbour spacing, all configurations on **one locked colour scale** —
unlocked, 14.5% clumping and 1.9% look identical. Ground truth is farthest-point
sampled, so its clumping is exactly 0.0%.

This is the picture behind the most robust result in the project: repulsion takes
clumping from 14.54% to 1.86% across five folds, every fold agreeing, t = -7.29.
None of it is visible in section 4.

Removed 2026-09-08: the signed-deviation panel. `mesh_viz.signed_deviation` fits a
plane to the 24 nearest ground-truth points, and that neighbourhood spans both
tables of the skull, so the plane lands mid-bone and the figure reports which
table a point is nearer rather than how far it is from its own surface — its
outside-percentage sits at 47-53% across eight very different skulls while the
true one ranges over 27-53%. Structurally blind, not imprecise. The replacement
is `src/eval/point_to_surface.py`, which uses the real mesh. The function and the
archived `dev_*` columns are kept, unchanged.

In [ ]:
items = [(gt[k_show], "ground truth")] + [(preds[n][k_show], n) for n, _ in MODELS]
mv.fig_spacing_grid(items, s_show,
                    title=f"skull {SKULL} — nearest-neighbour spacing "
                          f"(dark = clumped; colour scale locked)").show()

## 6 · Archive

Merged into `experiments_log/surface_quality.csv`, never overwritten: one of the
eleven rows already there was computed from weights that no longer exist anywhere.
Written columns are pinned to the ones the file already has.

Warning: `dev_*` is still computed and archived, and must not be quoted.

In [ ]:
rows = [(name, {k: float(np.mean([v[k] for v in per]))
                for k in per[0]})
        for name, per in ((name, [mv.surface_stats(preds[name][i], gt[i], float(scale_mm[i]))
                                  for i in sel])
                          for name, _ in MODELS)]

OUT = os.path.join(REPO, "experiments_log", "surface_quality.csv")
new = pd.DataFrame([{"model": n, **s} for n, s in rows])

if os.path.exists(OUT):
    prev = pd.read_csv(OUT)
    missing = set(prev.columns) - set(new.columns)
    assert not missing, f"the archived rows have columns this run does not produce: {missing}"
    new = new[list(prev.columns)]        # pin the schema, drop anything newly added
    merged = pd.concat([prev[~prev["model"].isin(new["model"])], new], ignore_index=True)
    print("replacing:", ", ".join(sorted(set(prev["model"]) & set(new["model"]))) or "(none)")
    print("adding:   ", ", ".join(sorted(set(new["model"]) - set(prev["model"]))) or "(none)")
    print("untouched:", ", ".join(sorted(set(prev["model"]) - set(new["model"]))) or "(none)")
else:
    merged = new

assert not merged["model"].duplicated().any(), "duplicate model rows -- the key did not match"
merged.to_csv(OUT, index=False)
print(f"\n-> {os.path.relpath(OUT, REPO)}  ({len(merged)} rows)")
print(f"\nthis run, {len(sel)} skulls in `ids` order:\n")
print(new[["model", "clump_pct", "spacing_cv", "spacing_median_mm"]].round(3).to_string(index=False))

## 7 · Appendix: reconstruction parameters

For exploring a *single* model. Never for comparing two — each knob changes how
smooth a surface looks, so tuning per figure lets a parameter impersonate a model
improvement.

| knob | what it does | range |
|---|---|---|
| `radius_mm` | ball radius grown around each point | 4-7. Below 4 the surface is a sponge, above 8 the defect closes |
| `sigma` | blur of the distance field, the main knob | 0-4 |
| `taubin` | mesh smoothing iterations, non-shrinking | 0-120 |
| `res` | distance-field grid, cost as res cubed | 96 fast, 160 detailed |

Ground truth gets the same ladder, because texture appearing there belongs to the
reconstruction, not to a model.

**Warning: `sigma` is in voxels, so lowering `res` doubles the physical blur.**
This ladder runs at `res=64` to keep the notebook small, half of `RECON`'s 128 —
which makes the `heavy` rung blur roughly twice as far in millimetres as its name
suggests. On a cranial vault, a thin shell around a large empty interior, that is
enough for the blurred distance field to rise above the isolevel and eat through
the bone: **the `heavy` panel shows a hole in the ground truth, which is complete.**
Measured on skull 039 — same points, same preset, only `res` differs: Euler
characteristic 0 at `res=64` (a hole right through) against 2 at `res=128`
(closed). The hole is the reconstruction, not the data and not the model.


In [ ]:
# res=64 rather than RECON's 128: each panel carries a whole mesh, and at 128 the
# "raw" rung alone is 370k faces -- four panels serialise to 30 MB, two figures put
# 60 MB into the notebook. This section is about relative texture, not detail.
LADDER_RES = 64

mv.fig_smoothing_ladder(preds[MESH_MODEL][k_show], s_show, res=LADDER_RES,
                        title=f"skull {SKULL} ({MESH_MODEL}) — smoothing ladder "
                              f"(exploration only, not for comparison)").show()

mv.fig_smoothing_ladder(gt[k_show], s_show, res=LADDER_RES,
                        title="ground truth — same ladder "
                              "(texture here belongs to the reconstruction)").show()